##

## Load Library and Data

In [ ]:
# install icd-mappings package
!pip install icd-mappings

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from icdmappings import Mapper
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, roc_auc_score
import re
from sklearn.compose import ColumnTransformer
import shap
import matplotlib.pyplot as plt

In [ ]:
# load the csv files into pandas dataframes
lab_events = pd.read_csv("lab_events.csv")
vital_signs = pd.read_csv("vital_signs.csv")
demographics = pd.read_csv("demographics.csv")
administrative = pd.read_csv("administrative.csv")
discharge_notes = pd.read_csv('discharge.csv')

In [ ]:
# check the first few rows of the lab events dataframe
lab_events.head()
# check the info of the lab events dataframe
lab_events.info()
# check duplicate rows in the lab events dataframe
lab_events.duplicated().sum()

In [ ]:
# check the first few rows of the vital signs dataframe
vital_signs.head()
# check the info of the vital signs dataframe
vital_signs.info()
# check duplicate rows in the vital signs dataframe
vital_signs.duplicated().sum()

In [ ]:
# check the first few rows of the demographics dataframe
demographics.head()
# check the info of the demographics dataframe
demographics.info()
# check duplicate rows in the demographics dataframe
demographics.duplicated().sum()
# delete the duplicate rows in demographics dataframe
demographics = demographics.drop_duplicates()

In [ ]:
# check the first few rows of the administrative dataframe
administrative.head()
# check the info of the administrative dataframe
administrative.info()
# check the duplicate rows in the administrative dataframe
administrative.duplicated().sum()

In [ ]:
# check the first few rows of the discharge notes dataframe
discharge_notes.head()
# check the info of the discharge notes dataframe
discharge_notes.info()
# check the duplicate rows in the discharge notes dataframe
discharge_notes.duplicated().sum()

In [ ]:
# merge four dataframes together on subject_id and hadm_id
merged_df = pd.merge(vital_signs, lab_events,on=["subject_id", "hadm_id", "stay_id", "intime"], how="inner")
print(merged_df.shape)
merged_df = pd.merge(merged_df, demographics, on=["subject_id", "hadm_id"], how="inner")
print(merged_df.shape)
merged_df = pd.merge(merged_df, administrative, on=["subject_id", "hadm_id", "stay_id"], how="inner")
print(merged_df.shape)
merged_df = pd.merge(merged_df, discharge_notes, on=["subject_id", "hadm_id"], how="inner")
print(merged_df.shape)

In [ ]:
merged_df.info()

## Filtering

In [ ]:
# check duplicate rows in merged dataframe
print(merged_df.duplicated().sum())
# check the shape of the merged dataframe
print(merged_df.shape)

In [ ]:
# keep only the patient that age >= 18
merged_df = merged_df[merged_df["age"] >= 18]
# count the number of rows in merged dataframe after keeping only the patient that age >= 18
merged_df.shape[0]

In [ ]:
# Onlt keep the patient that los >= 1 day
merged_df = merged_df[merged_df["los"] >= 1]
# count the number of rows in merged dataframe after keeping only the patient that los >= 1
merged_df.shape[0]

In [ ]:
# check the 99% quantile of the los column
los_99_quantile = merged_df["los"].quantile(0.99)
print(los_99_quantile)

# only keep the patient that los <= the 99% quantile of the los column
merged_df = merged_df[merged_df["los"] <= los_99_quantile]
print(merged_df.shape[0])

In [ ]:
# if the subject_id has multiple rows, keep the earliest one
merged_df = merged_df.sort_values(by=["subject_id", "intime"]).drop_duplicates(subset=["subject_id"], keep="first")
# count the number of rows in merged dataframe after keeping only the earliest row for each subject_id
merged_df.shape[0]

In [ ]:
# check for the percentage of missing values in each column of the merged dataframe
missing_values = merged_df.isnull().sum() / len(merged_df) * 100
print(missing_values)

### Blood Pressure Transformation

In [ ]:
# create a new column called "bp"
# "bp" column will be the mean of non_invasive_bp_mean column, but if non_invasive_bp_mean is null, then use art_bp_mean
merged_df["bp"] = merged_df["non_invasive_bp_mean"].fillna(merged_df["art_bp_mean"])

In [ ]:
# check for the percentage of missing values in each column of the merged dataframe
missing_values = merged_df.isnull().sum() / len(merged_df) * 100
print(missing_values)

In [ ]:
# drop columnns: art_bp_systolic, art_bp_diastolic, art_bp_mean, non_invasive_bp_systolic, non_invasive_bp_diastolic, non_invasive_bp_mean
merged_df = merged_df.drop(columns=["art_bp_systolic", "art_bp_diastolic", "art_bp_mean", "non_invasive_bp_systolic", "non_invasive_bp_diastolic", "non_invasive_bp_mean"])

In [ ]:
# drop the missing values in the merged dataframe
merged_df = merged_df.dropna()
merged_df.shape

### ICD Mapping

In [ ]:
# convert the idc 9 code to icd 10 code using the icdmappings package
mapper = Mapper()

# create two new columns to store the unified ICD code and version
merged_df['icd_code_unified'] = merged_df['icd_code']
merged_df['icd_version_unified'] = merged_df['icd_version']

# build a boolean mask to identify rows where the ICD version is 9
mask = merged_df['icd_version'] == 9

# extract the ICD-9 codes from the rows that match the mask, and convert them to a list
icd9_codes = merged_df.loc[mask, 'icd_code'].astype(str).str.strip().tolist()

# use the mapper to convert the list of ICD-9 codes to ICD-10 codes
mapped_icd10_codes = mapper.map(icd9_codes, source='icd9', target='icd10')

# update the new "icd_code_unified" column with the mapped ICD-10 codes for the rows that match the mask
merged_df.loc[mask, 'icd_code_unified'] = mapped_icd10_codes

# update the new "icd_version_unified" column to 10 for the rows that match the mask
merged_df.loc[mask, 'icd_version_unified'] = 10

# check the first few rows of the merged dataframe to see if the ICD-9 codes have been successfully converted to ICD-10 codes
print(merged_df[merged_df['icd_version'] == 9][['icd_code', 'icd_version', 'icd_code_unified', 'icd_version_unified']].head())

In [ ]:
# convert the unified ICD-10 codes to 22 chapters
# convert the unified ICD-10 codes to a list of strings, and remove any leading or trailing whitespace
icd10_codes = merged_df['icd_code_unified'].astype(str).str.strip().tolist()

icd10_chapters = mapper.map(icd10_codes, source='icd10', target='chapter')

# store the mapped ICD-10 chapters in a new column called "icd10_chapter"
merged_df['icd10_chapter'] = icd10_chapters

# check the first few rows of the merged dataframe to see if the ICD-10 codes have been successfully mapped to chapters
print(merged_df[['icd_code_unified', 'icd10_chapter']].head(10))

In [ ]:
# check for the missing values in the icd10_chapter column
missing_icd10_chapter = merged_df['icd10_chapter'].isnull().sum()
print(f"Number of missing values in icd10_chapter column: {missing_icd10_chapter}")

# drop the rows with missing values in the icd10_chapter column
merged_df = merged_df.dropna(subset=['icd10_chapter'])

# count the number of rows in the merged dataframe after dropping the rows with missing values in the icd10_chapter column
print(merged_df.shape[0])

### LOS Conversion

In [ ]:
# check the los description of the merged dataframe
merged_df["los"].describe()

In [ ]:
# convert the los column to category column with 2 categories: 0 for los <= 3 and 1 for los > 3
merged_df["los_category"] = merged_df["los"].apply(lambda x: 0 if x <= 3 else 1)

In [ ]:
# check the distribution of the los_category column
los_category_distribution = merged_df["los_category"].value_counts(normalize=True) * 100
print(los_category_distribution) 

In [ ]:
merged_df.info()

In [ ]:
# drop all the irrelevant columns that will not be used for modeling
merged_df = merged_df.drop(columns=["subject_id","hadm_id","stay_id", "intime", "icd_code", "icd_version", "icd_code_unified", "icd_version_unified", "note_id", "note_type", "note_seq", "charttime", "storetime", "los"])

# check the shape of the merged dataframe after dropping the irrelevant columns
print(merged_df.shape)

In [ ]:
# store the merged dataframe to a csv file
# merged_df.to_csv("merged_df.csv", index=False)

## Clean Note

In [ ]:
# load the merged dataframe from the csv file
# merged_df = pd.read_csv("merged_df.csv")

In [ ]:
# reindex the merged dataframe
merged_df = merged_df.reset_index(drop=True)

In [ ]:
# check an example of the text data in the discharge notes
print(merged_df["text"][0])

In [ ]:
# build a function to clean the text data
def clean_text(text):
    # Regex: Remove headers AND their values
    text = re.sub(r'(Name|Unit No|Admission Date|Discharge Date|Date of Birth|Sex|Attending):\s*[^\n]*\n?', ' ', text, flags=re.IGNORECASE)

    #Regex: Remove specific section headers (but keep the text under them)
    # there is no content under "Followup Instructions" in the discharge notes
    text = re.sub(r'(Followup Instructions):\s*', ' ', text, flags=re.IGNORECASE)
    # remove extra spaces
    text = re.sub(r'___+', '', text)
    # Remove non-alphanumeric characters except for basic punctuation
    text = re.sub(r'[^\w\s.,!?;:]', '', text)
    # replace multiple spaces with a single space and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# apply the clean_text function to the text column of the merged dataframe
merged_df['cleaned_text'] = merged_df['text'].apply(clean_text)

In [ ]:
# check an example of the text data after cleaning
print(merged_df["cleaned_text"][0])

In [ ]:
# drop the original text column from the merged dataframe
final_df =  merged_df.drop(columns=["text"])

# save the final dataframe to a csv file
#final_df.to_csv("final_df.csv", index=False)

## Get the embeddings with BioClinical ModernBERT

In [ ]:
# read the final dataframe from the csv file
# final_df = pd.read_csv("final_df.csv")

In [ ]:
# build a function to get the BERT embeddings for the cleaned text data in the dataframe
def get_bert_embeddings(texts, tokenizer, model, device, batch_size=8, max_length=8192):
    all_embeddings = []

    # define a mean pooling function to get the sentence embedding from the token embeddings
    def mean_pooling(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = (last_hidden_state * mask).sum(1)
        counts = mask.sum(1).clamp(min=1)
        return summed / counts

    # use tqdm to show the progress of encoding the texts
    pbar = tqdm(range(0, len(texts), batch_size), desc="Encoding")

    for i in pbar:
        batch_texts = [str(t) for t in texts[i : i + batch_size]]

        # encode the batch of texts using the tokenizer, and move the tensors to the specified device
        encoded = tokenizer(
            batch_texts,
            padding= True, # pad the sequences to the maximum length
            truncation=True, # truncate the sequences to the maximum length
            max_length=max_length,
            return_tensors="pt", # return PyTorch tensors
        )
        # move the encoded tensors to the specified device (CPU or GPU)
        encoded = {k: v.to(device) for k, v in encoded.items()}

        # get the token embeddings from the BERT model, without computing gradients
        with torch.no_grad():
            outputs = model(**encoded)

        # get the last hidden state from the model outputs, and apply mean pooling to get the sentence embeddings
        last_hidden = outputs.last_hidden_state
        embeddings = mean_pooling(last_hidden, encoded["attention_mask"])

        all_embeddings.append(embeddings.cpu().numpy())

        # show the current progress (number of texts processed) in the tqdm progress bar
        pbar.set_postfix({
            "done": min(i + batch_size, len(texts))
        })
    # convert the list of embeddings to a numpy array and return it
    return np.vstack(all_embeddings)

In [ ]:
# BioClinical-ModernBERT-base tokenizer and model
model_id = "thomas-sounack/BioClinical-ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
bioclinical_modernbert = AutoModel.from_pretrained(model_id)
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
bioclinical_modernbert = bioclinical_modernbert.to(device)
bioclinical_modernbert.eval()
print(f"Using device: {device}")

In [ ]:
# get embeddings for the sample data
print("Starting BioClinical-ModernBERT-base embedding generation...")
bert_embeddings = get_bert_embeddings(
    final_df['cleaned_text'], tokenizer, bioclinical_modernbert, device, max_length=8192, batch_size=4
)
print(f"Embeddings shape: {bert_embeddings.shape}")

In [ ]:
# save the embeddings numpy file
# np.save("BioClinical_ModernBERT_embeddings.npy", bert_embeddings)

In [ ]:
# read the bert embedding from the numpy file
# bert_embeddings = np.load("BioClinical_ModernBERT_embeddings.npy")

In [ ]:
# Combine structured data + BERT embeddings
# Drop both "text" and "text_clean" since they've been replaced by BERT embeddings
embedding_cols = [f"bert_{i}" for i in range(bert_embeddings.shape[1])]
clinicalbert_df = pd.DataFrame(bert_embeddings, columns=embedding_cols, index=final_df.index)
final_df_embeddings = pd.concat([final_df.drop(columns=["cleaned_text"]), clinicalbert_df], axis=1)
print(f"final_df_embeddings shape: {final_df_embeddings.shape}")
final_df_embeddings.head()

In [ ]:
# store the final dataframe with embeddings to a csv file
# final_df_embeddings.to_csv("final_df_embeddings.csv", index=False)

## Prepare the Data for Training

In [ ]:
# read the final dataframe with embeddings from the csv file
# final_df_embeddings = pd.read_csv("final_df_embeddings.csv")

In [ ]:
# categorical columns: ['gender', 'race', 'admission_type', 'admission_location', 'icd10_chapter']

feature_columns = [col for col in final_df_embeddings.columns if col not in ["los_category"]]
# define the categorical and numerical columns
categorical_columns = ['gender', 'race', 'admission_type', 'admission_location', 'icd10_chapter']

numerical_columns = [col for col in feature_columns if col not in categorical_columns]

# exclude the BERT embedding columns from the numerical columns
numerical_columns = [col for col in numerical_columns if not col.startswith("bert_")]

# bert embedding columns
bert_embedding_columns = [col for col in feature_columns if col.startswith("bert_")]

In [ ]:
# check the number of categorical columns, numerical columns, and bert embedding columns
print(f"Number of categorical columns: {len(categorical_columns)}")
print(f"Number of numerical columns: {len(numerical_columns)}")
print(f"Number of BERT embedding columns: {len(bert_embedding_columns)}")

In [ ]:
# print categorical and numerical columns
print(f"Categorical columns: {categorical_columns}")
print(f"Numerical columns: {numerical_columns}")

In [ ]:
X = final_df_embeddings.drop(columns=["los_category"])
y = final_df_embeddings["los_category"]

In [ ]:
# train test split

# first split the dataset into training and test sets, with 80% of the data for training and 20% for testing, and stratify by the target variable to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# split the training set into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# print the shapes of the training, validation, and test sets
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

In [ ]:
# do the one hot encoding for the categorical features, and keep the numerical features and BERT embedding features unchanged
# Create a ColumnTransformer to apply OneHotEncoder to the categorical features
preprocessor = ColumnTransformer(
    transformers=[
        # Let the numerical features pass through so they stay at the front of the DataFrame
        ('num', 'passthrough', numerical_columns),
        # Place the One-Hot Encoder next so newly created columns are appended at the end
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_columns)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# Force scikit-learn to output Pandas DataFrames
preprocessor.set_output(transform="pandas")

# Fit the preprocessor on the training data only, then transform the training, validation, and testing sets.
X_train = preprocessor.fit_transform(X_train)
X_val   = preprocessor.transform(X_val)
X_test  = preprocessor.transform(X_test)

In [ ]:
# standard scaling for numerical variables
scaler = StandardScaler()
X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

## MultiModal Models

### Traditional Machine Learning Models

#### Logistic Regression

In [ ]:
# Logistic Regression
# train logistic regression model
lr_model = LogisticRegression(class_weight='balanced', max_iter=2000, solver='saga',random_state=42)
lr_model.fit(X_train, y_train)

# evaluate the model on the test set
y_pred = lr_model.predict(X_test)
print("Logistic Regression")
print(classification_report(y_test, y_pred, digits=4))

# print auc roc score
y_prob = lr_model.predict_proba(X_test)[:, 1]
auc_roc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC Score For Logistic Regression: {auc_roc:.4f}")

In [ ]:
# random forest model
rf_model = RandomForestClassifier(
    n_estimators  = 500,
    max_depth     = 20,
    min_samples_leaf = 5,
    max_features  = 'sqrt',
    class_weight  = 'balanced',
    random_state  = 42,
    n_jobs        = -1,
    verbose       = 1
)

# fit the random forest model on the training data
rf_model.fit(X_train, y_train)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# evaluate the random forest model on the test set
print("Random Forest")
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf, digits=4))
print(f"AUC-ROC Score For Random Forest: {roc_auc_score(y_test, y_prob_rf):.4f}")

In [ ]:
# XGBoost
# handle class imbalance by calculating the scale_pos_weight parameter
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
scale_pos_weight = class_weights[1] / class_weights[0]
print(f"scale_pos_weight: {scale_pos_weight:.4f}")


xgb_parameters = {'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'early_stopping_rounds': 20,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 1}

xgb_model = XGBClassifier(**xgb_parameters)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

print("XGBoost")
print(classification_report(y_test, y_pred_xgb, digits=4))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob_xgb):.4f}")

### Deep Learning Model - MLP

In [ ]:
# MLP
# handle class imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

# build the MLP model with 5 hidden layers and dropout for regularization
mlp_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

# compile the model with Adam optimizer, binary crossentropy loss, and metrics for accuracy and AUC
mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

# set up callbacks for early stopping and learning rate reduction on plateau
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=5,
        mode='max',
        min_lr=1e-6
    )
]

# train the model on the training data, with validation on the validation set, and using class weights to handle class imbalance
history = mlp_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# evaluate the model on the test set
y_prob_mlp = mlp_model.predict(X_test).flatten()
y_pred_mlp = (y_prob_mlp >= 0.5).astype('int32')

In [ ]:
# print the classification report and AUC-ROC score for the MLP model
print("MLP")
print(classification_report(y_test, y_pred_mlp, digits=4))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_prob_mlp):.4f}")

In [ ]:
# save the MLP model
# mlp_model.save("mlp_model.h5")

### SHAP

In [ ]:
# shap for logistic regression
explainer_lr = shap.LinearExplainer(lr_model, X_train)
shap_values_lr = explainer_lr.shap_values(X_test)


In [ ]:
# shap for XGBoost
explainer_xgb = shap.TreeExplainer(xgb_model, X_train)
shap_values_xgb = explainer_xgb.shap_values(X_test)

In [ ]:
# mlp_model = keras.models.load_model("mlp_model.h5")

In [ ]:
# shap for MLP
explainer_mlp = shap.DeepExplainer(mlp_model, X_train.sample(n=1000, random_state=42).values.astype(np.float32))
shap_values_mlp = explainer_mlp.shap_values(X_test.values.astype(np.float32))

# deal with shape
if isinstance(shap_values_mlp, list):
    shap_values_mlp = shap_values_mlp[0]
if shap_values_mlp.ndim == 3:
    shap_values_mlp = shap_values_mlp.squeeze(-1)

In [ ]:
# show the result of three shap bar plot
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# Logistic Regression
plt.sca(axes[0])
shap.summary_plot(
    shap_values_lr, X_test, 
    plot_type="bar", 
    show=False, 
    max_display=20,  # display top 20 features
    plot_size=None 
)
axes[0].set_title("(a) Logistic Regression", fontsize=13, fontweight='bold')
axes[0].set_xlabel("mean(|SHAP value|)", fontsize=11)

# XGBoost
plt.sca(axes[1])
shap.summary_plot(
    shap_values_xgb, X_test, 
    plot_type="bar", 
    show=False, 
    max_display=20,
    plot_size=None
)
axes[1].set_title("(b) XGBoost", fontsize=13, fontweight='bold')
axes[1].set_xlabel("mean(|SHAP value|)", fontsize=11)

# MLP
plt.sca(axes[2])
shap.summary_plot(
    shap_values_mlp, X_test, 
    plot_type="bar", 
    show=False, 
    max_display=20,
    plot_size=None
)
axes[2].set_title("(c) MLP", fontsize=13, fontweight='bold')
axes[2].set_xlabel("mean(|SHAP value|)", fontsize=11)

fig.suptitle("SHAP Feature Importance Across Models", fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('shap_bar_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# show the result of three SHAP beeswarm plot
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

# Logistic Regression
plt.sca(axes[0])
shap.summary_plot(shap_values_lr, X_test, show=False, max_display=20, plot_size=None)
axes[0].set_title("(a) Logistic Regression", fontsize=13, fontweight='bold')

# XGBoost
plt.sca(axes[1])
shap.summary_plot(shap_values_xgb, X_test, show=False, max_display=20, plot_size=None)
axes[1].set_title("(b) XGBoost", fontsize=13, fontweight='bold')

# MLP
plt.sca(axes[2])
shap.summary_plot(shap_values_mlp, X_test, show=False, max_display=20, plot_size=None)
axes[2].set_title("(c) MLP", fontsize=13, fontweight='bold')

fig.suptitle("SHAP Value Distribution Across Models", fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('shap_beeswarm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Structured Data Only Models

### Prepare Structured-only Dataset

In [ ]:
# load final_df (structured data + cleaned_text, no BERT embeddings)
struct_df = pd.read_csv("final_df.csv").drop(columns=["cleaned_text"])

categorical_columns_s = ['gender', 'race', 'admission_type', 'admission_location', 'icd10_chapter']
numerical_columns_s = [col for col in struct_df.columns if col not in categorical_columns_s + ["los_category"]]

print(f"Numerical columns ({len(numerical_columns_s)}): {numerical_columns_s}")
print(f"Categorical columns ({len(categorical_columns_s)}): {categorical_columns_s}")
print(f"Dataset shape: {struct_df.shape}")

In [ ]:
# train test split for the structured data only
X_s = struct_df.drop(columns=["los_category"])
y_s = struct_df["los_category"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.2, random_state=42, stratify=y_s)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(X_train_s, y_train_s, test_size=0.25, random_state=42, stratify=y_train_s)

print(f"Training set shape: {X_train_s.shape}")
print(f"Validation set shape: {X_val_s.shape}")
print(f"Test set shape: {X_test_s.shape}")

In [ ]:
# preprocess the structured data with one hot encoding for categorical features and standard scaling for numerical features

# one hot encoding for categorical features
preprocessor_s = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_columns_s),
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_columns_s)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)
preprocessor_s.set_output(transform="pandas")

X_train_s = preprocessor_s.fit_transform(X_train_s)
X_val_s   = preprocessor_s.transform(X_val_s)
X_test_s  = preprocessor_s.transform(X_test_s)

# standard scaling for numerical features
scaler_s = StandardScaler()
X_train_s[numerical_columns_s] = scaler_s.fit_transform(X_train_s[numerical_columns_s])
X_val_s[numerical_columns_s]   = scaler_s.transform(X_val_s[numerical_columns_s])
X_test_s[numerical_columns_s]  = scaler_s.transform(X_test_s[numerical_columns_s])

print(f"X_train_s shape after preprocessing: {X_train_s.shape}")

### Models (Structured Only)

In [ ]:
# Logistic Regression (structured only)
lr_model_s = LogisticRegression(class_weight='balanced', max_iter=2000, solver='saga', random_state=42)
lr_model_s.fit(X_train_s, y_train_s)

y_pred_lr_s = lr_model_s.predict(X_test_s)
y_prob_lr_s = lr_model_s.predict_proba(X_test_s)[:, 1]

print("Logistic Regression (Structured Only)")
print(classification_report(y_test_s, y_pred_lr_s, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test_s, y_prob_lr_s):.4f}")

In [ ]:
# Random Forest (structured only)
rf_model_s = RandomForestClassifier(
    n_estimators=500, max_depth=20, min_samples_leaf=5,
    max_features='sqrt', class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=1
)
rf_model_s.fit(X_train_s, y_train_s)

y_pred_rf_s = rf_model_s.predict(X_test_s)
y_prob_rf_s = rf_model_s.predict_proba(X_test_s)[:, 1]

print("Random Forest (Structured Only)")
print(classification_report(y_test_s, y_pred_rf_s, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test_s, y_prob_rf_s):.4f}")

In [ ]:
# XGBoost (structured only)
class_weights_s = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train_s)
scale_pos_weight_s = class_weights_s[1] / class_weights_s[0]

xgb_parameters_s = {
    'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5,
    'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'eval_metric': 'auc', 'scale_pos_weight': scale_pos_weight_s,
    'early_stopping_rounds': 20, 'random_state': 42, 'n_jobs': -1, 'verbosity': 1
}

xgb_model_s = XGBClassifier(**xgb_parameters_s)
xgb_model_s.fit(X_train_s, y_train_s, eval_set=[(X_val_s, y_val_s)], verbose=50)

y_prob_xgb_s = xgb_model_s.predict_proba(X_test_s)[:, 1]
y_pred_xgb_s = (y_prob_xgb_s >= 0.5).astype(int)

print("XGBoost (Structured Only)")
print(classification_report(y_test_s, y_pred_xgb_s, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test_s, y_prob_xgb_s):.4f}")

In [ ]:
# MLP (structured only)
class_weights_s_dict = {0: class_weights_s[0], 1: class_weights_s[1]}

mlp_model_s = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),

    layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(1, activation='sigmoid')
])

mlp_model_s.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

callbacks_s = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, mode='max', restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=5, mode='max', min_lr=1e-6)
]

history_s = mlp_model_s.fit(
    X_train_s, y_train_s,
    epochs=100, batch_size=64,
    validation_data=(X_val_s, y_val_s),
    class_weight=class_weights_s_dict,
    callbacks=callbacks_s, verbose=1
)

y_prob_mlp_s = mlp_model_s.predict(X_test_s).flatten()
y_pred_mlp_s = (y_prob_mlp_s >= 0.5).astype('int32')

print("MLP (Structured Only)")
print(classification_report(y_test_s, y_pred_mlp_s, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test_s, y_prob_mlp_s):.4f}")